In [22]:
!pip install pyspark

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, when
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Create a Spark session
spark = SparkSession.builder \
    .appName("Lab4_Spark_MLlib") \
    .master("local[*]") \
    .getOrCreate()

In [40]:
# Q1: Read all CSV files from tp4_data into a single DataFrame
df = spark.read.csv("/home/tp4_data/*.csv", header=True, inferSchema=True)
#since all the files have the same format we can just use inferSchema
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [25]:
# Q2: Display the schema
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [26]:
# Q3: Fill missing values (NaN) with 0
df = df.fillna(0)
df.show(5)


+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [47]:
# Q4: Add a "day_of_week" column with the day name (Monday, Tuesday, etc.)
df = df.withColumn("day_of_week", date_format(col("InvoiceDate"), "EEEE"))
df.show(20)
#The second argument in the date format is a Java date pattern.
#"EEEE" means:Full day name

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|day_of_week|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|     Monday|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|     Monday|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|     Monday|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|     Monday|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|     Monday|
|   537226|    22705|   WRAP GREEN PEARS

In [49]:
# Q5: Split data based on InvoiceDate
train_df = df.filter(col("InvoiceDate") < "2010-12-13")
test_df = df.filter(col("InvoiceDate") >= "2010-12-13")

print(f"Training set count: {train_df.count()}")
print(f"Test set count: {test_df.count()}")

Training set count: 26732
Test set count: 18676


In [29]:
# Q6: Create a StringIndexer for the "day_of_week" column
# StringIndexer converts categorical string values into numerical indices
indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_index")
indexer

StringIndexer_03087d6841a5

In [30]:
# Q7: Use OneHotEncoder to resolve the ordinal issue
# OneHotEncoder converts index values into binary vectors, removing implicit ordering
encoder = OneHotEncoder(inputCol="day_of_week_index", outputCol="day_of_week_encoded")
#note that we applied the one hot encoding on the day of the week index
#this would be passed later in the pipeline
encoder

OneHotEncoder_549dbf0621e3

In [51]:
# Q8: Create a VectorAssembler combining UnitPrice, Quantity, and day_of_week_encoded
#VectorAssembler takes several numeric / vector columns
#and packs them into ONE vector column that ML algorithms can understand.
#as Spark ML models accept only one input column for features:
#for example :
# UnitPrice = 3.5
# Quantity  = 2
# day_of_week_encoded = [0,0,0,1,0,0,0]
# features = [3.5, 2, 0,0,0,1,0,0,0]

assembler = VectorAssembler(
    inputCols=["UnitPrice", "Quantity", "day_of_week_encoded"],
    outputCol="features"
)
assembler


VectorAssembler_1ab0b5ed9318

In [32]:
# Q9: Create a Pipeline with StringIndexer -> OneHotEncoder -> VectorAssembler
pipeline = Pipeline(stages=[indexer, encoder, assembler])
pipeline

Pipeline_7f150809454e

## Question 10: How does StringIndexer handle unseen values during transformation?

**Answer:** By default, StringIndexer throws an error when it encounters values in the test/transform data that were not seen during training (fitting).

To handle this, we can set the `handleInvalid` parameter:
- `"error"` (default): throws an exception for unseen labels
- `"skip"`: filters out rows with unseen labels
- `"keep"`: assigns unseen labels a new index (the last index + 1)

We use `"keep"` so unseen day names in the test set are handled gracefully.

In [33]:
# Q10: Set handleInvalid="keep" on StringIndexer to handle unseen labels in the test set
indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_index", handleInvalid="keep")

# Rebuild the pipeline with the updated indexer
pipeline = Pipeline(stages=[indexer, encoder, assembler])
pipeline

Pipeline_f74c5aec5b21

In [ ]:
# -----------------------------------------------
# What does .fit() do?
# -----------------------------------------------
# pipeline.fit(train_df)
#
# - "fit" TRAINs all Estimators inside the pipeline on train_df.
# - Estimators = stages that need to learn from data.
#   Examples:
#       - StringIndexer  → learns label → index mapping
#       - OneHotEncoder  → learns category size
#       - ML model       → learns model parameters (if included)
#
# - Output of .fit() is a PipelineModel.
# - After fitting, all Estimators become Transformers.
#
# Important:
# - fit() DOES learning.
# - It does NOT modify the original DataFrame.


# -----------------------------------------------
# What does .transform() do?
# -----------------------------------------------
# pipeline_model.transform(train_df)
#
# - "transform" APPLIES the learned transformations to the dataset.
# - It adds new columns (index, encoded, features, predictions, etc.).
# - It does NOT learn anything new.
#
# In short:
#   fit()      → learn from data
#   transform() → apply what was learned


# -----------------------------------------------
# Does this use an ML algorithm by default?
# -----------------------------------------------
# NO.
#
# A Pipeline does NOT automatically include a machine learning model.
# It only runs the stages you explicitly added.
#
# If your pipeline only contains:
#   - StringIndexer
#   - OneHotEncoder
#   - VectorAssembler
#
# Then it is ONLY doing feature engineering.
# There is NO classifier or regressor involved.
#
# Spark ML does NOT choose any algorithm automatically.


# -----------------------------------------------
# When does an ML algorithm get used?
# -----------------------------------------------
# Only if you explicitly add one, for example:
#
#   from pyspark.ml.regression import LinearRegression
#   lr = LinearRegression(featuresCol="features", labelCol="TotalPrice")
#
#   pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
#
# In that case:
#   fit()  → trains LinearRegression
#   transform() → produces predictions
#
# Without a model stage → no ML algorithm is used.


# -----------------------------------------------
# Summary:
# -----------------------------------------------
# fit()       → trains/learns parameters from training data
# transform() → applies learned transformations
# Default ML algorithm → NONE (must be explicitly added)

In [55]:
# Q11: Fit the pipeline on the training data and transform it
pipeline_model = pipeline.fit(train_df)
train_transformed = pipeline_model.transform(train_df)
train_transformed.select("UnitPrice", "Quantity", "day_of_week", "day_of_week_index", "day_of_week_encoded", "features").show(5, truncate=False)
#the output of the day of the week encoded is a sparse vector of args :
#(size, [indices], [values])
#ex : (6,[4],[1.0])
#meaning it is of size 6 and the value at index 4 is 1.0 and all others are 03
#note that it is of size 6 and not 7 becasue by default : OneHotEncoder(dropLast=True)
#features is a single vector that contains ALL inputs your ML model will use.
#ex:(8,[0,1,6],[2.95,6.0,1.0])
#we have 8 cols in the original df and val of col 0 is 2.95 and col 1 is 6.0 and so on

+---------+--------+-----------+-----------------+-------------------+---------------------------+
|UnitPrice|Quantity|day_of_week|day_of_week_index|day_of_week_encoded|features                   |
+---------+--------+-----------+-----------------+-------------------+---------------------------+
|2.95     |6       |Monday     |4.0              |(6,[4],[1.0])      |(8,[0,1,6],[2.95,6.0,1.0]) |
|2.1      |8       |Monday     |4.0              |(6,[4],[1.0])      |(8,[0,1,6],[2.1,8.0,1.0])  |
|5.95     |2       |Monday     |4.0              |(6,[4],[1.0])      |(8,[0,1,6],[5.95,2.0,1.0]) |
|1.65     |6       |Monday     |4.0              |(6,[4],[1.0])      |(8,[0,1,6],[1.65,6.0,1.0]) |
|0.42     |25      |Monday     |4.0              |(6,[4],[1.0])      |(8,[0,1,6],[0.42,25.0,1.0])|
+---------+--------+-----------+-----------------+-------------------+---------------------------+
only showing top 5 rows


In [35]:
# Q12: Create a KMeans instance with 20 clusters
kmeans = KMeans(k=20, featuresCol="features", predictionCol="prediction", seed=42)
kmeans

KMeans_50e6064d5b76

In [ ]:
# Q13: Train the KMeans model on the transformed training data
kmeans_model = kmeans.fit(train_transformed)


# Display the cluster centers
print("Cluster Centers:")
for i, center in enumerate(kmeans_model.clusterCenters()):
    print(f"  Cluster {i}: {center}")

Cluster Centers:
  Cluster 0: [ 1.86355192 15.29856759  0.2580573   0.2385855   0.17547001  0.12757386
  0.1103402   0.08997314]
  Cluster 1: [ 1.3524695e+04 -5.0000000e-01  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00]
  Cluster 2: [ 3.00e-02 -9.36e+03  0.00e+00  1.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00]
  Cluster 3: [ 6.70671e+03 -1.00000e+00  0.00000e+00  0.00000e+00  0.00000e+00
  0.00000e+00  0.00000e+00  1.00000e+00]
  Cluster 4: [8.86762e+02 1.00000e+00 0.00000e+00 2.00000e-01 3.00000e-01 0.00000e+00
 3.00000e-01 2.00000e-01]
  Cluster 5: [1.80e-01 2.88e+03 0.00e+00 1.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00]
  Cluster 6: [2.00702247e+00 1.00831461e+02 2.41573034e-01 2.19101124e-01
 2.92134831e-01 9.55056180e-02 6.74157303e-02 8.42696629e-02]
  Cluster 7: [3.600e-01 1.776e+03 0.000e+00 5.000e-01 0.000e+00 5.000e-01 0.000e+00
 0.000e+00]
  Cluster 8: [1.0800e+00 1.4305e+03 0.0000e+00 7.5000e-01 2.5000e-01 0.0000e+00
 0.0000e

In [ ]:
# Q14: Transform the test set using the pipeline model, then predict with KMeans
test_transformed = pipeline_model.transform(test_df)
predictions = kmeans_model.transform(test_transformed)

predictions.select("InvoiceNo", "Description", "UnitPrice", "Quantity", "day_of_week", "features", "prediction").show(10, truncate=False)

+---------+-----------------------------------+---------+--------+-----------+---------------------------+----------+
|InvoiceNo|Description                        |UnitPrice|Quantity|day_of_week|features                   |prediction|
+---------+-----------------------------------+---------+--------+-----------+---------------------------+----------+
|539325   |SET OF 3 CAKE TINS PANTRY DESIGN   |4.95     |3       |Friday     |(8,[0,1,4],[4.95,3.0,1.0]) |11        |
|539325   |SET OF 6 SPICE TINS PANTRY DESIGN  |3.95     |4       |Friday     |(8,[0,1,4],[3.95,4.0,1.0]) |11        |
|539325   |ASSORTED BOTTLE TOP  MAGNETS       |0.42     |12      |Friday     |(8,[0,1,4],[0.42,12.0,1.0])|0         |
|539325   |FRIDGE MAGNETS US DINER ASSORTED   |0.85     |12      |Friday     |(8,[0,1,4],[0.85,12.0,1.0])|0         |
|539325   |FRIDGE MAGNETS LES ENFANTS ASSORTED|0.85     |12      |Friday     |(8,[0,1,4],[0.85,12.0,1.0])|0         |
|539325   |ENGLISH ROSE GARDEN SECATEURS      |0.85     

In [ ]:
# Q15: Calculate the Silhouette coefficient using ClusteringEvaluator
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette_score = evaluator.evaluate(predictions)
print(f"Silhouette Score (with squared Euclidean distance): {silhouette_score}")

: 